In [1]:
import os
import numpy as np
import cv2
import glob
import torch
from torch.utils.data import Dataset
import mediapipe as mp
import json
from tqdm import tqdm

Data Loading and Preparation
Create a PyTorch dataset class to load your preprocessed landmarks and prepare them for training

In [2]:
class LandmarkDataset(Dataset):
    def __init__(self, landmarks_dir, vocab_file=None):
        self.landmarks_dir = landmarks_dir
        self.samples = sorted(glob.glob(os.path.join(landmarks_dir, "*.npz")))

        # Build vocabulary from glosses
        self.gloss_vocab = self._build_vocab() if vocab_file is None else self._load_vocab(vocab_file)

    def _build_vocab(self):
        glosses = set()
        for sample_path in self.samples:
            data = np.load(sample_path, allow_pickle=True)
            glosses.update(data['glosses'])
        gloss_to_idx = {gloss: idx+2 for idx, gloss in enumerate(sorted(glosses))}
        gloss_to_idx['<pad>'] = 0
        gloss_to_idx['<blank>'] = 1
        return gloss_to_idx

    def __getitem__(self, idx):
        data = np.load(self.samples[idx], allow_pickle=True)
        landmarks = torch.FloatTensor(data['landmarks'])
        glosses = [self.gloss_vocab[g] for g in data['glosses']]
        return landmarks, torch.LongTensor(glosses)


Implement collate function for variable-length sequences

In [3]:
def collate_fn(batch):
    landmarks, glosses = zip(*batch)

    # Pad landmarks to max length in batch
    max_len = max(lm.shape[0] for lm in landmarks)
    padded_landmarks = torch.zeros(len(landmarks), max_len, landmarks[0].shape[1])
    landmark_lengths = []

    for i, lm in enumerate(landmarks):
        padded_landmarks[i, :lm.shape[0]] = lm
        landmark_lengths.append(lm.shape[0])

    # Pad glosses
    max_gloss_len = max(len(g) for g in glosses)
    padded_glosses = torch.zeros(len(glosses), max_gloss_len).long()
    gloss_lengths = []

    for i, g in enumerate(glosses):
        padded_glosses[i, :len(g)] = g
        gloss_lengths.append(len(g))

    return padded_landmarks, torch.LongTensor(landmark_lengths), \
           padded_glosses, torch.LongTensor(gloss_lengths)


Model Architecture
Implement a Transformer-based architecture with CTC loss for alignment-free training :​

In [4]:
from torch import nn


class SignLanguageTranslator(nn.Module):
    def __init__(self, input_dim, num_glosses, d_model=512, nhead=8,
                 num_encoder_layers=6, num_decoder_layers=6):
        super().__init__()

        # Feature embedding
        self.input_proj = nn.Linear(input_dim, d_model)

        # Spatial-temporal encoder (for continuous features)
        self.temporal_conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)

        # Transformer encoder for video features
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=2048,
            dropout=0.1, activation='relu', batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

        # CTC head for gloss recognition
        self.ctc_head = nn.Linear(d_model, num_glosses)

        # Transformer decoder for translation (if doing gloss-to-text)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=2048,
            dropout=0.1, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)
        self.output_head = nn.Linear(d_model, num_glosses)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model)

    def forward(self, src, src_lengths, tgt=None):
        # Embed input
        x = self.input_proj(src)  # [B, T, d_model]
        x = self.pos_encoder(x)

        # Temporal convolution
        x_conv = self.temporal_conv(x.transpose(1, 2)).transpose(1, 2)
        x = x + x_conv

        # Create mask for padding
        src_mask = self._generate_padding_mask(src_lengths, src.size(1))

        # Encode
        memory = self.encoder(x, src_key_padding_mask=src_mask)

        # CTC prediction for recognition
        ctc_logits = self.ctc_head(memory)

        if tgt is not None:
            # Decoder for translation
            tgt_emb = self.pos_encoder(self.input_proj(tgt))
            tgt_mask = self._generate_square_subsequent_mask(tgt.size(1))
            output = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                                 memory_key_padding_mask=src_mask)
            output = self.output_head(output)
            return ctc_logits, output

        return ctc_logits

    def _generate_padding_mask(self, lengths, max_len):
        batch_size = len(lengths)
        mask = torch.arange(max_len).expand(batch_size, max_len) >= lengths.unsqueeze(1)
        return mask.to(lengths.device)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                            -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


Training Pipeline
Implement the training loop with CTC loss :

In [5]:
import torch.nn.functional as F
from torch.nn import CTCLoss

def train_model(model, train_loader, val_loader, num_epochs=50):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    ctc_loss = CTCLoss(blank=1, zero_infinity=True)
    ce_loss = nn.CrossEntropyLoss(ignore_index=0)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4,
                                 betas=(0.9, 0.98), eps=1e-9)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch_idx, (landmarks, landmark_lengths, glosses, gloss_lengths) in enumerate(train_loader):
            landmarks = landmarks.to(device)
            glosses = glosses.to(device)

            optimizer.zero_grad()

            # Forward pass
            ctc_logits = model(landmarks, landmark_lengths)

            # CTC loss expects [T, B, C] format
            ctc_logits = ctc_logits.transpose(0, 1).log_softmax(2)

            # Calculate CTC loss
            loss = ctc_loss(ctc_logits, glosses, landmark_lengths, gloss_lengths)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")

        # Validation
        val_loss = validate(model, val_loader, ctc_loss, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch} - Train Loss: {total_loss/len(train_loader):.4f}, "
              f"Val Loss: {val_loss:.4f}")


Data Augmentation
Apply temporal augmentation techniques :

In [6]:
class TemporalAugmentation:
    def __init__(self, speed_range=(0.8, 1.2)):
        self.speed_range = speed_range

    def __call__(self, landmarks):
        # Random temporal scaling
        speed = np.random.uniform(*self.speed_range)
        num_frames = landmarks.shape[0]
        new_num_frames = int(num_frames / speed)

        indices = np.linspace(0, num_frames-1, new_num_frames)
        augmented = np.array([landmarks[int(i)] for i in indices])

        return augmented

# Add spatial noise
def add_spatial_noise(landmarks, noise_std=0.01):
    noise = np.random.normal(0, noise_std, landmarks.shape)
    return landmarks + noise


Evaluation Metrics
Implement Word Error Rate (WER) for evaluation :

In [7]:
def compute_wer(predictions, targets):
    from jiwer import wer
    return wer(targets, predictions)

def decode_predictions(ctc_output, vocab):
    # CTC beam search decoding
    from torch.nn.functional import log_softmax

    decoded = []
    for sample in ctc_output:
        # Simple greedy decoding
        pred = torch.argmax(sample, dim=-1)
        # Remove blanks and repeated tokens
        pred_seq = []
        prev = None
        for p in pred:
            if p != 1 and p != prev:  # 1 is blank
                pred_seq.append(p.item())
            prev = p
        decoded.append([vocab[p] for p in pred_seq])
    return decoded


Full example

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn import CTCLoss
import glob
import json
import math
from tqdm import tqdm

# ==================== DATASET ====================

class LandmarkDataset(Dataset):
    """Dataset loader for preprocessed landmarks"""
    def __init__(self, landmarks_dir, vocab_file=None, build_vocab=False):
        self.landmarks_dir = landmarks_dir
        self.samples = sorted(glob.glob(os.path.join(landmarks_dir, "*.npz")))

        if build_vocab:
            self.gloss_vocab = self._build_vocab()
            self._save_vocab(vocab_file)
        else:
            self.gloss_vocab = self._load_vocab(vocab_file)

        self.idx_to_gloss = {v: k for k, v in self.gloss_vocab.items()}

    def _build_vocab(self):
        """Build vocabulary from all glosses in dataset"""
        glosses = set()
        for sample_path in tqdm(self.samples, desc="Building vocab"):
            data = np.load(sample_path, allow_pickle=True)
            glosses.update(data['glosses'])

        gloss_to_idx = {'<pad>': 0, '<blank>': 1, '<sos>': 2, '<eos>': 3}
        for idx, gloss in enumerate(sorted(glosses)):
            gloss_to_idx[gloss] = idx + 4

        print(f"Vocabulary size: {len(gloss_to_idx)}")
        return gloss_to_idx

    def _save_vocab(self, vocab_file):
        """Save vocabulary to JSON file"""
        if vocab_file:
            with open(vocab_file, 'w') as f:
                json.dump(self.gloss_vocab, f, indent=2)
            print(f"Vocabulary saved to {vocab_file}")

    def _load_vocab(self, vocab_file):
        """Load vocabulary from JSON file"""
        with open(vocab_file, 'r') as f:
            vocab = json.load(f)
        print(f"Vocabulary loaded from {vocab_file}, size: {len(vocab)}")
        return vocab

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        data = np.load(self.samples[idx], allow_pickle=True)
        landmarks = torch.FloatTensor(data['landmarks'])
        glosses = [self.gloss_vocab[str(g)] for g in data['glosses']]
        return landmarks, torch.LongTensor(glosses)


def collate_fn(batch):
    """Collate function for variable-length sequences"""
    landmarks, glosses = zip(*batch)

    # Pad landmarks to max length in batch
    max_len = max(lm.shape[0] for lm in landmarks)
    feature_dim = landmarks[0].shape[1]
    padded_landmarks = torch.zeros(len(landmarks), max_len, feature_dim)
    landmark_lengths = []

    for i, lm in enumerate(landmarks):
        padded_landmarks[i, :lm.shape[0]] = lm
        landmark_lengths.append(lm.shape[0])

    # Pad glosses
    max_gloss_len = max(len(g) for g in glosses)
    padded_glosses = torch.zeros(len(glosses), max_gloss_len).long()
    gloss_lengths = []

    for i, g in enumerate(glosses):
        padded_glosses[i, :len(g)] = g
        gloss_lengths.append(len(g))

    return (padded_landmarks, torch.LongTensor(landmark_lengths),
            padded_glosses, torch.LongTensor(gloss_lengths))


# ==================== MODEL ====================

class PositionalEncoding(nn.Module):
    """Positional encoding for transformer"""
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                            -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class SignLanguageTranslator(nn.Module):
    """Transformer-based Sign Language Translation Model"""
    def __init__(self, input_dim, num_glosses, d_model=512, nhead=8,
                 num_encoder_layers=6, dropout=0.1):
        super().__init__()

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout)
        )

        # Temporal convolution for local feature extraction
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='relu',
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

        # CTC head for gloss prediction
        self.ctc_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_glosses)
        )

        self.d_model = d_model

    def forward(self, src, src_lengths):
        # Input projection: [B, T, input_dim] -> [B, T, d_model]
        x = self.input_proj(src)

        # Temporal convolution
        x_conv = self.temporal_conv(x.transpose(1, 2)).transpose(1, 2)
        x = x + x_conv

        # Positional encoding
        x = self.pos_encoder(x)

        # Create padding mask
        src_mask = self._generate_padding_mask(src_lengths, src.size(1)).to(src.device)

        # Encode
        memory = self.encoder(x, src_key_padding_mask=src_mask)

        # CTC prediction
        ctc_logits = self.ctc_head(memory)

        return ctc_logits

    def _generate_padding_mask(self, lengths, max_len):
        """Generate padding mask for variable length sequences"""
        batch_size = len(lengths)
        mask = torch.arange(max_len).expand(batch_size, max_len) >= lengths.unsqueeze(1)
        return mask


# ==================== DATA AUGMENTATION ====================

class TemporalAugmentation:
    """Temporal data augmentation"""
    def __init__(self, speed_range=(0.85, 1.15), prob=0.5):
        self.speed_range = speed_range
        self.prob = prob

    def __call__(self, landmarks):
        if np.random.random() > self.prob:
            return landmarks

        # Random temporal scaling
        speed = np.random.uniform(*self.speed_range)
        num_frames = landmarks.shape[0]
        new_num_frames = max(1, int(num_frames / speed))

        indices = np.linspace(0, num_frames - 1, new_num_frames)
        augmented = np.array([landmarks[min(int(i), num_frames-1)] for i in indices])

        return augmented


def add_spatial_noise(landmarks, noise_std=0.005):
    """Add Gaussian noise to landmarks"""
    noise = np.random.normal(0, noise_std, landmarks.shape)
    return landmarks + noise


# ==================== TRAINING ====================

def decode_predictions(ctc_output, lengths, vocab, blank_id=1):
    """Decode CTC output using greedy decoding"""
    batch_size = ctc_output.size(0)
    predictions = torch.argmax(ctc_output, dim=-1)

    decoded = []
    for i in range(batch_size):
        pred = predictions[i, :lengths[i]]

        # Remove blanks and repeated tokens
        pred_seq = []
        prev = None
        for p in pred:
            p = p.item()
            if p != blank_id and p != prev:
                pred_seq.append(p)
            prev = p

        # Convert to glosses
        glosses = [vocab.get(p, '<unk>') for p in pred_seq]
        decoded.append(glosses)

    return decoded


def compute_wer(predictions, targets):
    """Compute Word Error Rate"""
    errors = 0
    total_words = 0

    for pred, tgt in zip(predictions, targets):
        # Simple edit distance calculation
        pred_words = pred if isinstance(pred, list) else pred.split()
        tgt_words = tgt if isinstance(tgt, list) else tgt.split()

        d = [[0] * (len(tgt_words) + 1) for _ in range(len(pred_words) + 1)]

        for i in range(len(pred_words) + 1):
            d[i][0] = i
        for j in range(len(tgt_words) + 1):
            d[0][j] = j

        for i in range(1, len(pred_words) + 1):
            for j in range(1, len(tgt_words) + 1):
                if pred_words[i-1] == tgt_words[j-1]:
                    d[i][j] = d[i-1][j-1]
                else:
                    d[i][j] = min(d[i-1][j], d[i][j-1], d[i-1][j-1]) + 1

        errors += d[len(pred_words)][len(tgt_words)]
        total_words += len(tgt_words)

    return errors / max(total_words, 1)


def train_epoch(model, train_loader, optimizer, criterion, device, vocab):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    num_batches = 0

    pbar = tqdm(train_loader, desc="Training")
    for batch_idx, (landmarks, landmark_lengths, glosses, gloss_lengths) in enumerate(pbar):
        landmarks = landmarks.to(device)
        glosses = glosses.to(device)
        landmark_lengths = landmark_lengths.to(device)
        gloss_lengths = gloss_lengths.to(device)

        optimizer.zero_grad()

        # Forward pass
        ctc_logits = model(landmarks, landmark_lengths)

        # CTC loss expects [T, B, C] format and log probabilities
        ctc_logits = ctc_logits.transpose(0, 1)
        log_probs = F.log_softmax(ctc_logits, dim=-1)

        # Calculate CTC loss
        loss = criterion(log_probs, glosses, landmark_lengths, gloss_lengths)

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / num_batches


def validate(model, val_loader, criterion, device, vocab, idx_to_gloss):
    """Validate the model"""
    model.eval()
    total_loss = 0
    num_batches = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation")
        for landmarks, landmark_lengths, glosses, gloss_lengths in pbar:
            landmarks = landmarks.to(device)
            glosses = glosses.to(device)
            landmark_lengths = landmark_lengths.to(device)
            gloss_lengths = gloss_lengths.to(device)

            # Forward pass
            ctc_logits = model(landmarks, landmark_lengths)

            # Calculate loss
            ctc_logits_t = ctc_logits.transpose(0, 1)
            log_probs = F.log_softmax(ctc_logits_t, dim=-1)
            loss = criterion(log_probs, glosses, landmark_lengths, gloss_lengths)

            total_loss += loss.item()
            num_batches += 1

            # Decode predictions
            predictions = decode_predictions(ctc_logits, landmark_lengths, idx_to_gloss)

            # Convert targets to glosses
            for i in range(glosses.size(0)):
                target = glosses[i, :gloss_lengths[i]].cpu().numpy()
                target_glosses = [idx_to_gloss.get(int(t), '<unk>') for t in target]
                all_predictions.append(predictions[i])
                all_targets.append(target_glosses)

            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / num_batches
    wer = compute_wer(all_predictions, all_targets)

    return avg_loss, wer


def train_model(model, train_loader, val_loader, num_epochs, device, vocab, idx_to_gloss, save_dir='checkpoints'):
    """Full training loop"""
    os.makedirs(save_dir, exist_ok=True)

    criterion = CTCLoss(blank=1, zero_infinity=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3)

    best_wer = float('inf')

    for epoch in range(num_epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"{'='*50}")

        # Train
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, vocab)
        print(f"Train Loss: {train_loss:.4f}")

        # Validate
        val_loss, val_wer = validate(model, val_loader, criterion, device, vocab, idx_to_gloss)
        print(f"Val Loss: {val_loss:.4f}, Val WER: {val_wer:.4f}")

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save best model
        if val_wer < best_wer:
            best_wer = val_wer
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_wer': val_wer,
            }
            torch.save(checkpoint, os.path.join(save_dir, 'best_model.pt'))
            print(f"✓ Saved best model with WER: {best_wer:.4f}")

        # Save checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_wer': val_wer,
            }
            torch.save(checkpoint, os.path.join(save_dir, f'checkpoint_epoch_{epoch+1}.pt'))

    return best_wer


# ==================== MAIN ====================

def main():
    # Configuration
    LANDMARKS_TRAIN = "./landmarks_train"
    LANDMARKS_DEV = "./landmarks_dev"
    VOCAB_FILE = "vocab.json"
    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    INPUT_DIM = 1659  # 126 (hands) + 1434 (face) + 99 (pose)
    D_MODEL = 512
    NHEAD = 8
    NUM_LAYERS = 6
    DROPOUT = 0.1

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Create datasets
    print("\n" + "="*50)
    print("Loading Datasets")
    print("="*50)

    train_dataset = LandmarkDataset(
        LANDMARKS_TRAIN,
        vocab_file=VOCAB_FILE,
        build_vocab=True
    )

    val_dataset = LandmarkDataset(
        LANDMARKS_DEV,
        vocab_file=VOCAB_FILE,
        build_vocab=False
    )
    use_pin_memory = torch.cuda.is_available()
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=use_pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=use_pin_memory
    )

    # Create model
    print("\n" + "="*50)
    print("Creating Model")
    print("="*50)

    num_glosses = len(train_dataset.gloss_vocab)
    model = SignLanguageTranslator(
        input_dim=INPUT_DIM,
        num_glosses=num_glosses,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Train model
    print("\n" + "="*50)
    print("Starting Training")
    print("="*50)

    best_wer = train_model(
        model,
        train_loader,
        val_loader,
        NUM_EPOCHS,
        device,
        train_dataset.gloss_vocab,
        train_dataset.idx_to_gloss
    )

    print(f"\n{'='*50}")
    print(f"Training Complete! Best WER: {best_wer:.4f}")
    print(f"{'='*50}")


if __name__ == "__main__":
    main()


Using device: cpu

Loading Datasets


Building vocab: 100%|██████████| 5672/5672 [00:01<00:00, 4223.42it/s]
C:\Projects\SignNet\.venv\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Vocabulary size: 1235
Vocabulary saved to vocab.json
Vocabulary loaded from vocab.json, size: 1235

Creating Model
Total parameters: 21,972,691
Trainable parameters: 21,972,691

Starting Training

Epoch 1/50


Training:   0%|          | 0/355 [00:00<?, ?it/s]C:\Projects\SignNet\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
